# Land-Variable Lead-Time nRMSE Skill Maps

This is the land RMSE companion to `1a_lnd_leadtime_acc_skill_map.ipynb` and to
`1b_atm_leadtime_rmse_skill_map.ipynb`/`1b_ocn_leadtime_rmse_skill_map.ipynb`. It
reuses the same cached skill product `1a_lnd` writes (normalized RMSE is already computed
alongside ACC by `esp_lab.stats.compute_skill_seasonal`), so this notebook adds
the RMSE-focused map presentation. Compatible ACC skill caches are reused; missing or incompatible caches are computed using the same land skill workflow.

Prepared inputs and skills use the same source-first hierarchy as `1a`:
`<S2D_DIAG_ROOT>/<source-or-case>/leadtime_acc/{inputs,skill}/land/<field>/`.


In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import display

from esp_lab import land_input_cache, land_skill, stats
from esp_lab import env_paths
from esp_lab.leadtime_plot_utils import (
    add_lead_badge,
    add_missing_map_panel,
    seasonal_label,
    style_global_map_axis,
)
from esp_lab.paths import leadtime_acc_dir
from esp_lab.utils.filename_utils import safe_token
from esp_lab.utils.netcdf_utils import atomic_to_netcdf, load_netcdf
from esp_lab.utils.dask_util import DaskConfig, get_cluster_client
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources


## Managed Dask resources

A local production run uses worker processes with the shared-node safety cap.
Rerunning this cell closes any previous notebook cluster and tracked datasets first.


In [ ]:
import dask
from dask.distributed import (
    wait,
    get_client
)
from esp_lab.utils.notebook_resources import restart_notebook_cluster, close_notebook_resources

dask.__version__

# User-adjustable Dask settings (shared login nodes are capped at four workers).
DASK_SETTINGS = {
    "enabled": True,
    "cluster_type": "local",
    "workers": 12,
    "memory_limit": "4GB",
}

dask_cfg = DaskConfig(
    cluster_type=DASK_SETTINGS["cluster_type"],
    workers=DASK_SETTINGS["workers"],
    memory_limit=DASK_SETTINGS["memory_limit"],
)
cluster, client, workflow_resources = restart_notebook_cluster(
    globals(),
    lambda: get_cluster_client(dask_cfg) if DASK_SETTINGS["enabled"] else (None, None),
)
if client is not None:
    display(client)

## User setup

Choose the land field, archives, evaluation period, cache policy, Dask resources, and
outputs here. `WORKFLOW_SETTINGS["run"]` groups the experiment and resource
choices, while `WORKFLOW_SETTINGS["regrid"]` controls target resolution, method,
and periodicity. `prepared_land.mode="auto"` with inventory identity is the normal
production setting. After an inventory run, `source_identity_mode="snapshot"` with
`prepared_land.mode="require"` provides an archive-free restart from the exact prepared
inputs. `source_identity_mode="prefer_snapshot"` is the fast default: it uses
saved source identities when available and falls back to a fresh inventory only
for inputs that have not been snapshotted yet. `smoke_mode=True` selects a small
but archive-backed two-case run. Switch back to `inventory` after raw files are
updated in place so their sizes and modification times are checked again.


In [ ]:
# Select H2OSNO, H2OSOI, or TWS.
field = "H2OSNO" #"H2OSOI" #"TWS"
# Optional common processing end year. None uses the end year from
# run.initialization_years.
YEAR_END = 2011  # None
SOIL_DEPTH_RANGE_M = (0.0, 1.6) if field == "H2OSOI" else None

# Experiment used as the control in the nRMSE-difference figure.
# Every other configured experiment is plotted as experiment minus control.
RMSE_DIFFERENCE_CONTROL = "E3SM-Reanalysis"

WORKFLOW_SETTINGS = {
    "run": {
        "smoke_mode": False,
        "initialization_years": (1980, 2018 if YEAR_END is None else int(YEAR_END)),
        "year_end": YEAR_END,
        "init_months": [5, 11],
        "climatology_years": (1981, 2010),
        "ensemble_members": [f"EN{i:02d}" for i in range(10)],
        "monthly_nlead": 24,
        "seasonal_nlead": 8,
        "detrend": True,
        # False matches 1a's fast path: file/time/member labels are checked, but
        # native-grid values are not scanned twice before regridding and writing.
        # Set True for the expensive, cell-by-cell completeness audit.
        "strict_member_completeness": False,
        "force_compute": False,
        "raw_model_chunks": {"Y": 3, "L": 24, "M": 2, "lat": 90, "lon": 180},
        "model_chunks": {"Y": -1, "L": 4, "M": 2, "lat": 45, "lon": 90},
        "reference_chunks": {"time": -1, "lat": 45, "lon": 90},
    },
    "regrid": {
        "target_dlat": 5.0,
        "target_dlon": 5.0,
        "method": "conservative",
        "periodic": True,
    },
}
RUN = WORKFLOW_SETTINGS["run"]
REGRID = WORKFLOW_SETTINGS["regrid"]
GRID_TAG = land_input_cache.target_grid_tag(
    REGRID["target_dlat"], REGRID["target_dlon"]
)

E3SM_CASES = {
    "E3SM-Reanalysis": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_BruteForce",
        "cache_tag": "Reanalysis",
        "display_name": "E3SMv3-Reanalysis",
        "source_revision": "post_process_v1",
    },
    "E3SM-FOSIRL": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
        "cache_tag": "JRA55_FOSIRL",
        "display_name": "E3SMv3-FOSIRL",
        "source_revision": "post_process_v1",
    },
    "E3SM-4DEnVarOcn": {
        "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
        "cache_tag": "4DEnVarOcn",
        "display_name": "E3SMv3-4DEnVarOcn",
        "source_revision": "post_process_v1",
    },
}

REFERENCE_CONFIGS = {
    "H2OSNO": {
        "path": f"{env_paths.data_root()}/C3S_SWE/1x1/monthly/swe_*.nc",
        "variable": "swe", "product": "C3S_SWE",
        "documentation": "Copernicus Climate Change Service snow water equivalent",
        "source_revision": "c3s_swe_archive_2026-08-26",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": False, "already_seasonal": False,
        "mask_negative_categorical_flags": True,
        "require_complete_calendar_months": True,
        "retain_missing_seasons": True,
        "restrict_model_to_reference_months": True,
        "require_complete_evaluation_cohort": True,
    },
    "TWS": {
        "path": f"{env_paths.data_root()}/C3S_TWSA/1x1/monthly/twsa_*.nc",
        "variable": "twsa", "product": "C3S_TWSA",
        "documentation": "C3S terrestrial water storage anomaly v1.0; doi:10.5880/GFZ.C3S_TWSA_v1.0",
        "source_revision": "c3s_twsa_v1.0_archive_2026-09-09",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": True, "already_seasonal": False,
        "require_complete_calendar_months": True,
        "retain_missing_seasons": True,
        "restrict_model_to_reference_months": True,
        "require_complete_evaluation_cohort": False,
        "minimum_evaluation_years": 7,
    },
    "H2OSOI": {
        "path": f"{env_paths.data_root()}/CPC_SOM/monthly/soilw_*.nc",
        "variable": "soilw", "product": "CPC_Soil_Moisture_V2",
        "documentation": "https://www.cpc.ncep.noaa.gov/soilmst/descrip.htm",
        "source_revision": "cpc_soil_moisture_v2_archive_2026-08-26",
        "scale": 1.0, "offset": 0.0, "output_units": "mm",
        "is_anomaly": False, "already_seasonal": False,
        "vertical_dim": None, "layer_bounds_variable": None,
        "represented_depth_range_m": (0.0, 1.6),
        "restrict_model_to_reference_months": False,
        "require_complete_evaluation_cohort": True,
    },
}
reference_cfg = REFERENCE_CONFIGS[field]
REFERENCE_PRODUCT = reference_cfg["product"]
if REFERENCE_PRODUCT == "CONFIGURE_ME":
    raise ValueError("Configure the TWS reference product and raw-data contract")

CACHE_SETTINGS = {
    "prepared_land": {"mode": "auto"},  # auto, rebuild, or require
    "source_identity_mode": "prefer_snapshot",  # prefer_snapshot, inventory, snapshot, or revision
    "cleanup_temp_files": True,
    "temp_file_max_age_hours": 24.0,
}
PATHS = {
    "raw_model_root": str(env_paths.raw_model_root()),
    "staged_input_root": str(env_paths.s2d_diag_root()),
    "cache_output_root": str(env_paths.s2d_diag_root()),
    "figure_outdir": str(env_paths.figure_root()),
}
if RUN["smoke_mode"]:
    RUN.update(
        initialization_years=(2000, 2004), year_end=None, init_months=[11],
        climatology_years=(2000, 2004),
        ensemble_members=["EN00", "EN01"], monthly_nlead=6, seasonal_nlead=2,
    )
    print("ESP-Lab land smoke mode: two cases, one month, five years, two members/leads.")

run_year_start, run_year_end = map(int, RUN["initialization_years"])
configured_year_end = RUN.get("year_end")
processing_year_end = (
    run_year_end if configured_year_end is None else int(configured_year_end)
)
if not run_year_start <= processing_year_end <= run_year_end:
    raise ValueError(
        f"run.year_end {processing_year_end} must lie within configured "
        f"initialization years {run_year_start}-{run_year_end}"
    )
PROCESSING_YEARS = (run_year_start, processing_year_end)
if "E3SM-4DEnVarOcn" in E3SM_CASES and processing_year_end > 2011:
    raise ValueError(
        f"E3SM-4DEnVarOcn only spans 1980-2011. For multi-model comparison with 4DEnVar, "
        f"set YEAR_END = 2011 or remove 'E3SM-4DEnVarOcn' from E3SM_CASES."
    )
print(
    f"Processing initialization years: "
    f"{PROCESSING_YEARS[0]}-{PROCESSING_YEARS[1]}"
)

RAW_MODEL_ROOT = Path(PATHS["raw_model_root"])
STAGED_INPUT_ROOT = Path(PATHS["staged_input_root"])
S2D_DIAG_ROOT = Path(PATHS["cache_output_root"])
FIGURE_OUTDIR = Path(PATHS["figure_outdir"])
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

EVALUATION_PROTOCOL = (
    f"land_acc_init{PROCESSING_YEARS[0]}-{PROCESSING_YEARS[1]}_"
    f"clim{RUN['climatology_years'][0]}-{RUN['climatology_years'][1]}_"
    f"per-lead_{'detrend' if RUN['detrend'] else 'nodetrend'}"
)

REFERENCE_DISPLAY_NAMES = {
    "CPC_Soil_Moisture_V2": "CPC V2",
    "C3S_SWE": "C3S SWE",
    "C3S_TWSA": "C3S TWSA",
}
REFERENCE_DISPLAY_NAME = REFERENCE_DISPLAY_NAMES.get(
    REFERENCE_PRODUCT, REFERENCE_PRODUCT.replace("_", " "),
)
if field == "H2OSOI":
    if REFERENCE_PRODUCT == "CPC_Soil_Moisture_V2":
        REFERENCE_LEVEL_DEPTH = "0–1.6m"
        REFERENCE_CAPTION_NAME = "0–1.6m soil moisture"
    elif REFERENCE_PRODUCT == "C3S_SWE":
        REFERENCE_LEVEL_DEPTH = "0-5cm"
        REFERENCE_CAPTION_NAME = "upper ~5 cm"

init_month_names = {
    1: "JAN", 2: "FEB", 3: "MAR", 4: "APR", 5: "MAY", 6: "JUN",
    7: "JUL", 8: "AUG", 9: "SEP", 10: "OCT", 11: "NOV", 12: "DEC",
}
MAP_STYLE = {
    "lon_ticks": [-160, -80, 0, 80, 160],
    "lat_ticks": [-60, -30, 0, 30, 60],
    "tick_length": 2.5, "tick_width": 0.5,
    "grid_linewidth": 0.35, "grid_color": "0.35",
    "grid_alpha": 0.35,
} 

# Stable output names: rerunning a field replaces its previous figures.
FIGURE_FILENAMES = {
    "H2OSNO": {
        "rmse": "fig_lnd_rmse_h2osno_rmse.png",
        "difference": "fig_lnd_rmse_h2osno_rmse_difference.png",
    },
    "TWS": {
        "rmse": "fig_lnd_rmse_tws_rmse.png",
        "difference": "fig_lnd_rmse_tws_rmse_difference.png",
    },
    "H2OSOI": {
        "rmse": "fig_lnd_rmse_h2osoi_rmse.png",
        "difference": "fig_lnd_rmse_h2osoi_rmse_difference.png",
    },
}
field_fig_names = FIGURE_FILENAMES.get(field, {
    "rmse": f"fig_lnd_rmse_{field.lower()}_rmse.png",
    "difference": f"fig_lnd_rmse_{field.lower()}_rmse_difference.png",
})

FIGURE_SETTINGS = {
    "rmse": {
        "filename": field_fig_names["rmse"],
        "max_leads": 7,
        "cmap": "YlOrRd",
        "color_interval": 0.1, "color_min": 0.5, "color_max": 1.5,
        "color_cutoff": 1.0,
        "font_size": 18,
        "axis_label_font_scale": 0.75, "lead_label_font_scale": 0.75,
        "title_font_scale": 1.20, "colorbar_tick_font_scale": 0.80,
        "figsize": (26.4, 19.6),  # width, height in inches
        "layout_rect": (0.0, 0.075, 1.0, 0.96),
        "subplot_bottom": 0.09, "subplot_hspace": 0.06,
        "subplot_wspace": 0.025, "section_gap": 0.035,
        "month_header_y": 0.962, "divider_y0": 0.08,
        "divider_y1": 0.95, "divider_color": "0.25",
        "divider_linewidth": 1.0, "font_weight": "bold",
        "colorbar_rect": (0.28, 0.025, 0.44, 0.015),
        "colorbar_label": "nRMSE", "colorbar_orientation": "horizontal",
        "title_y": 0.995, "dpi": 300,
    },
    "difference": {
        "filename": field_fig_names["difference"],
        "color_interval": 0.05, "color_min": -0.5, "color_max": 0.5,
        "color_cutoff": 0.25, "cmap": "blue2red",
        "figsize": (16.0, 17.8),  # matches the RMSE panel aspect ratio
        "font_size": 18,
        "title_template": "{variable} nRMSE differences: experiments minus {control}, {trend} (ref: {reference})",
        "colorbar_label": r"${\Delta}$nRMSE",
        "title_font_scale": 1.1, "month_header_font_scale": 1.0,
        "axis_label_font_scale": 0.75, "lead_label_font_scale": 0.75,
        "colorbar_tick_font_scale": 0.8, "title_y": 0.992,
        "layout_edges": (0.035, 0.10, 0.995, 0.91),
        "month_header_y": 0.945, "divider_y0": 0.085,
        "divider_y1": 0.91, "divider_color": "0.25",
        "divider_linewidth": 1.0, "font_weight": "bold",
        "colorbar_orientation": "horizontal", "layout_pad": 0.3,
        "layout_h_pad": 0.15, "layout_w_pad": 0.1,
        "subplot_top": 0.91, "subplot_bottom": 0.10,
        "subplot_hspace": 0.025, "subplot_wspace": 0.02,
        "section_gap": 0.04,
        "colorbar_rect": (0.30, 0.025, 0.40, 0.014), "dpi": 300,
    },
}

## Prepare or reuse analysis-ready inputs

Planning is independent for the reference and every model case/month. Inventory mode
records exact file identities and snapshots before any expensive preparation. Snapshot
mode never resolves raw paths and requires compatible prepared caches. Dataset contracts
validate dimensions, members, years, grid, units, depth treatment, and provenance.


In [ ]:
prepared_mode = CACHE_SETTINGS["prepared_land"]["mode"]
if prepared_mode not in {"auto", "rebuild", "require"}:
    raise ValueError("prepared_land.mode must be 'auto', 'rebuild', or 'require'")
source_identity_mode = CACHE_SETTINGS["source_identity_mode"]
if source_identity_mode not in {"prefer_snapshot", "inventory", "snapshot", "revision"}:
    raise ValueError("source_identity_mode must be prefer_snapshot, inventory, snapshot, or revision")
if source_identity_mode == "snapshot" and prepared_mode != "require":
    raise ValueError("Snapshot mode requires prepared_land.mode='require'")

if "workflow_resources" not in globals():
    raise RuntimeError("Run the managed Dask setup cell before preparation")
netcdf_write_options = {
    "cleanup_temporary": CACHE_SETTINGS["cleanup_temp_files"],
    "temp_file_max_age_hours": CACHE_SETTINGS["temp_file_max_age_hours"],
}
years = np.arange(PROCESSING_YEARS[0], PROCESSING_YEARS[1] + 1)
target_grid = land_input_cache.target_grid(
    REGRID["target_dlat"], REGRID["target_dlon"]
)
SOURCE_SNAPSHOT_DIR = S2D_DIAG_ROOT / "tmp" / "source_inventory_snapshots" / "land"
if not SOURCE_SNAPSHOT_DIR.exists() and (S2D_DIAG_ROOT / "source_inventory_snapshots" / "land").exists():
    SOURCE_SNAPSHOT_DIR = S2D_DIAG_ROOT / "source_inventory_snapshots" / "land"
DEPTH_TOKEN = land_skill.land_depth_token(field, SOIL_DEPTH_RANGE_M)
reference_path = land_skill.staged_land_input_path(
    STAGED_INPUT_ROOT, REFERENCE_PRODUCT, field, GRID_TAG,
    depth_range_m=SOIL_DEPTH_RANGE_M,
)
model_input_paths = {
    case_name: {
        init_month: land_skill.staged_land_input_path(
            STAGED_INPUT_ROOT, case_cfg["cache_tag"], field, GRID_TAG,
            init_month=init_month, depth_range_m=SOIL_DEPTH_RANGE_M,
        )
        for init_month in RUN["init_months"]
    }
    for case_name, case_cfg in E3SM_CASES.items()
}

reference_files = []
reference_identity_kwargs = {
    "logical_identity": {
        "product": REFERENCE_PRODUCT, "variable": reference_cfg["variable"],
        "field": field, "path_pattern": reference_cfg["path"],
    },
    "source_revision": reference_cfg["source_revision"],
    "inventory_root": Path(reference_cfg["path"]).parent,
    "snapshot_dir": SOURCE_SNAPSHOT_DIR,
}
if source_identity_mode == "prefer_snapshot":
    try:
        reference_raw_identity = land_input_cache.raw_source_identity(
            paths=[], mode="snapshot", **reference_identity_kwargs
        )
        print("Using saved source identity for reference")
    except (RuntimeError, ValueError):
        reference_files = land_input_cache.resolve_reference_files(reference_cfg["path"])
        reference_raw_identity = land_input_cache.raw_source_identity(
            paths=reference_files, mode="inventory", **reference_identity_kwargs
        )
        print("Created source-identity snapshot for reference")
else:
    if source_identity_mode == "inventory":
        reference_files = land_input_cache.resolve_reference_files(reference_cfg["path"])
    reference_raw_identity = land_input_cache.raw_source_identity(
        paths=reference_files, mode=source_identity_mode, **reference_identity_kwargs
    )
reference_month_policy = (
    "complete centered 3-month windows; retain explicit missing seasons"
    if reference_cfg.get("require_complete_calendar_months", False)
    else "all seasonal center months"
)
reference_expected_attrs = land_input_cache.expected_attrs(
    field=field, source_kind="reference", source_name=REFERENCE_PRODUCT,
    source_data_identity=reference_raw_identity,
    initialization_years=PROCESSING_YEARS,
    climatology_years=RUN["climatology_years"],
    ensemble_members=RUN["ensemble_members"], monthly_nlead=RUN["monthly_nlead"],
    target_grid_name=GRID_TAG, regridding_method=REGRID["method"],
    regridding_periodic=REGRID["periodic"],
    output_units=reference_cfg["output_units"],
    reference_is_anomaly=reference_cfg["is_anomaly"],
    depth_range_m=SOIL_DEPTH_RANGE_M,
    reference_month_policy=reference_month_policy,
)
reference_ok, reference_reason = land_input_cache.prepared_cache_status(
    reference_path, reference_expected_attrs, grid=target_grid
)
if prepared_mode == "require" and not reference_ok:
    raise FileNotFoundError(f"Required prepared reference is unavailable: {reference_reason}")
if prepared_mode == "rebuild" or not reference_ok:
    if not reference_files:
        reference_files = land_input_cache.resolve_reference_files(reference_cfg["path"])
    print(f"Preparing reference ({reference_reason}): {reference_path}")
    land_input_cache.prepare_reference_cache(
        paths=reference_files, cfg=reference_cfg, field=field,
        depth_range_m=SOIL_DEPTH_RANGE_M, grid=target_grid,
        periodic=REGRID["periodic"],
        expected=reference_expected_attrs, output_path=reference_path,
        write_options=netcdf_write_options,
    )
else:
    print("Reusing prepared reference:", reference_path)

reference_dataset = xr.open_dataset(reference_path, chunks=RUN["reference_chunks"])
workflow_resources.track(reference_dataset)
land_input_cache.validate_dataset(
    reference_dataset, reference_expected_attrs, grid=target_grid
)
reference = reference_dataset[field]
reference_is_anomaly = reference_cfg["is_anomaly"]
reference_data_identity = land_input_cache.prepared_data_identity(
    reference_expected_attrs
)

model_expected_attrs = {}
model_data_identity_by_case_month = {}
for case_name, case_cfg in E3SM_CASES.items():
    model_expected_attrs[case_name] = {}
    model_data_identity_by_case_month[case_name] = {}
    for init_month in RUN["init_months"]:
        model_files = []
        model_identity_kwargs = {
            "logical_identity": {
                "case": case_name, "case_prefix": case_cfg["case_prefix"],
                "field": field, "init_month": init_month,
                "years": list(map(int, years)),
                "members": list(RUN["ensemble_members"]),
                "monthly_nlead": int(RUN["monthly_nlead"]),
            },
            "source_revision": case_cfg["source_revision"],
            "inventory_root": RAW_MODEL_ROOT,
            "snapshot_dir": SOURCE_SNAPSHOT_DIR,
        }
        if source_identity_mode == "prefer_snapshot":
            try:
                raw_identity = land_input_cache.raw_source_identity(
                    paths=[], mode="snapshot", **model_identity_kwargs
                )
                print(f"Using saved source identity: {case_name}, init={init_month}")
            except (RuntimeError, ValueError):
                model_files = land_input_cache.resolve_model_files(
                    data_dir=RAW_MODEL_ROOT, case_prefix=case_cfg["case_prefix"],
                    members=RUN["ensemble_members"], years=years,
                    init_month=init_month, field=field, nlead=RUN["monthly_nlead"],
                )
                raw_identity = land_input_cache.raw_source_identity(
                    paths=model_files, mode="inventory", **model_identity_kwargs
                )
                print(f"Created source-identity snapshot: {case_name}, init={init_month}")
        else:
            if source_identity_mode == "inventory":
                model_files = land_input_cache.resolve_model_files(
                    data_dir=RAW_MODEL_ROOT, case_prefix=case_cfg["case_prefix"],
                    members=RUN["ensemble_members"], years=years,
                    init_month=init_month, field=field, nlead=RUN["monthly_nlead"],
                )
            raw_identity = land_input_cache.raw_source_identity(
                paths=model_files, mode=source_identity_mode, **model_identity_kwargs
            )
        expected = land_input_cache.expected_attrs(
            field=field, source_kind="model", source_name=case_name,
            source_data_identity=raw_identity, case_prefix=case_cfg["case_prefix"],
            init_month=init_month, initialization_years=PROCESSING_YEARS,
            climatology_years=RUN["climatology_years"],
            ensemble_members=RUN["ensemble_members"], monthly_nlead=RUN["monthly_nlead"],
            target_grid_name=GRID_TAG, regridding_method=REGRID["method"],
            regridding_periodic=REGRID["periodic"],
            output_units=reference_cfg["output_units"],
            reference_is_anomaly=reference_cfg["is_anomaly"],
            depth_range_m=SOIL_DEPTH_RANGE_M,
            reference_month_policy=reference_month_policy,
        )
        path = model_input_paths[case_name][init_month]
        compatible, reason = land_input_cache.prepared_cache_status(
            path, expected, grid=target_grid
        )
        if prepared_mode == "require" and not compatible:
            raise FileNotFoundError(f"Required prepared model input {path} is unavailable: {reason}")
        if prepared_mode == "rebuild" or not compatible:
            print(f"Preparing {case_name}, init={init_month} ({reason}): {path}")
            land_input_cache.prepare_model_cache(
                data_dir=RAW_MODEL_ROOT, case_prefix=case_cfg["case_prefix"],
                members=RUN["ensemble_members"], years=years, init_month=init_month,
                field=field, monthly_nlead=RUN["monthly_nlead"],
                monthly_chunks=RUN["raw_model_chunks"],
                depth_range_m=SOIL_DEPTH_RANGE_M, reference=reference,
                restrict_to_reference_months=reference_cfg.get(
                    "restrict_model_to_reference_months", False
                ),
                climatology_years=RUN["climatology_years"],
                strict_member_completeness=RUN["strict_member_completeness"],
                grid=target_grid, expected=expected, output_path=path,
                periodic=REGRID["periodic"],
                write_options=netcdf_write_options,
                resolved_files=model_files or None,
            )
        else:
            print(f"Reusing prepared model input: {case_name}, init={init_month}: {path}")
        model_expected_attrs[case_name][init_month] = expected
        model_data_identity_by_case_month[case_name][init_month] = (
            land_input_cache.prepared_data_identity(expected)
        )

print(f"Prepared-input mode: {prepared_mode}; source identity: {source_identity_mode}")


In [ ]:
forecast_by_case_month = {}
valid_time_by_case_month = {}
expected_years_by_case_month = {}
for case_name, case_cfg in E3SM_CASES.items():
    forecast_by_case_month[case_name] = {}
    valid_time_by_case_month[case_name] = {}
    expected_years_by_case_month[case_name] = {}
    for init_month in RUN["init_months"]:
        path = model_input_paths[case_name][init_month]
        ds = xr.open_dataset(path, chunks=RUN["model_chunks"])
        workflow_resources.track(ds)
        land_input_cache.validate_dataset(
            ds, model_expected_attrs[case_name][init_month], grid=target_grid
        )
        forecast = ds[field]
        land_skill.validate_land_reference_compatibility(forecast, reference)
        forecast, valid_time, dropped_leads = land_skill.retain_valid_seasonal_leads(
            forecast, ds.time
        )
        valid_time = valid_time.load()
        expected_years = land_skill.validate_hindcast_evaluation_setup(
            forecast, valid_time, init_month=init_month,
            initialization_years=PROCESSING_YEARS,
            expected_members=RUN["ensemble_members"],
            climatology_years=RUN["climatology_years"],
            require_complete_member_grid=RUN["strict_member_completeness"],
        )
        forecast_by_case_month[case_name][init_month] = forecast
        valid_time_by_case_month[case_name][init_month] = valid_time
        expected_years_by_case_month[case_name][init_month] = expected_years
        print("Validated:", case_name, init_month, path, "dropped leads:", dropped_leads)

VALID_LEADS_BY_MONTH = {}
for init_month in RUN["init_months"]:
    lead_coordinates = {
        tuple(forecast_by_case_month[case][init_month].L.values.tolist())
        for case in E3SM_CASES
    }
    if len(lead_coordinates) != 1:
        raise ValueError(
            f"Models do not share valid leads for init={init_month}: {lead_coordinates}"
        )
    VALID_LEADS_BY_MONTH[init_month] = list(next(iter(lead_coordinates)))
print("Valid seasonal leads by initialization month:", VALID_LEADS_BY_MONTH)


## Establish common target-year cohorts

For each initialization month and lead, all E3SM cases use the same intersection of model and reference target years. This prevents apparent skill differences caused only by unequal temporal samples.

In [ ]:
common_years_by_month = {}
model_climatology_count_by_month = {}
for init_month in RUN["init_months"]:
    model_fields = {
        case_name: forecast_by_case_month[case_name][init_month]
        for case_name in E3SM_CASES
    }
    model_times = {
        case_name: valid_time_by_case_month[case_name][init_month]
        for case_name in E3SM_CASES
    }
    leads = VALID_LEADS_BY_MONTH[init_month][: RUN["seasonal_nlead"]]
    expected_cohorts = {
        case_name: {int(lead): expected_years_by_case_month[case_name][init_month][int(lead)] for lead in leads}
        for case_name in E3SM_CASES
    }
    first_expected = expected_cohorts[next(iter(expected_cohorts))]
    if any(cohort != first_expected for cohort in expected_cohorts.values()):
        raise ValueError(f"Cases have different configured target years for init={init_month}")
    common_years = stats.common_valid_target_years_seasonal(
        model_fields, model_times, reference, leads, require_all_members=True
    )
    require_complete_cohort = reference_cfg.get(
        "require_complete_evaluation_cohort", True
    )
    if require_complete_cohort and common_years != first_expected:
        raise ValueError(
            f"Evaluation cohort is incomplete for init={init_month}: "
            f"expected={first_expected}, common={common_years}"
        )
    minimum_years = int(reference_cfg.get("minimum_evaluation_years", 3))
    undersampled = {
        lead: years for lead, years in common_years.items()
        if len(years) < minimum_years
    }
    if undersampled:
        raise ValueError(
            f"Reference leaves fewer than {minimum_years} evaluation years "
            f"for init={init_month}: {undersampled}"
        )
    land_skill.validate_reference_time_coverage(
        reference, common_years, model_times[next(iter(model_times))],
        RUN["climatology_years"],
        reference_is_anomaly=reference_is_anomaly,
    )
    common_years_by_month[init_month] = common_years
    climy0, climy1 = RUN["climatology_years"]
    model_climatology_count_by_month[init_month] = {
        lead: sum(climy0 <= year <= climy1 for year in years)
        for lead, years in common_years.items()
    }
    print(init_month, {lead: (min(years), max(years), len(years)) for lead, years in common_years.items()})
    print("model climatology N by lead:", model_climatology_count_by_month[init_month])

## Reuse or compute shared ACC and normalized-RMSE metrics

In [ ]:
climy0, climy1 = RUN["climatology_years"]
trend_tag = "detrend" if RUN["detrend"] else "nodetrend"
reference_cache_token = safe_token(REFERENCE_PRODUCT)
skill_by_case_month = {}
skill_paths_by_case_month = {}

for case_name, case_cfg in E3SM_CASES.items():
    skill_by_case_month[case_name] = {}
    skill_paths_by_case_month[case_name] = {}
    outdir = leadtime_acc_dir(
        case_cfg["cache_tag"], "skill", "land", field, root=S2D_DIAG_ROOT
    )
    outdir.mkdir(parents=True, exist_ok=True)
    for init_month in RUN["init_months"]:
        cohorts = common_years_by_month[init_month]
        cohort_token = land_skill.land_cohort_token(cohorts)
        depth_part = f"_{DEPTH_TOKEN}" if DEPTH_TOKEN else ""
        filename = (
            f"{case_cfg['cache_tag']}{init_month:02d}_{field}{depth_part}_"
            f"{reference_cache_token}_skill_{cohort_token}_"
            f"clim_{climy0}_{climy1}_{safe_token(GRID_TAG)}_{trend_tag}.nc"
        )
        outfile = outdir / filename
        skill_paths_by_case_month[case_name][init_month] = outfile
        expected_attrs = land_skill.expected_land_skill_attrs(
            field=field, init_month=init_month,
            climatology_years=RUN["climatology_years"],
            initialization_years=PROCESSING_YEARS,
            ensemble_members=RUN["ensemble_members"],
            target_years_by_lead=cohorts,
            reference_product=REFERENCE_PRODUCT,
            reference_data_identity=reference_data_identity,
            model_data_identity=model_data_identity_by_case_month[case_name][init_month],
            target_grid=GRID_TAG, detrend=RUN["detrend"],
            evaluation_protocol=EVALUATION_PROTOCOL,
            depth_range_m=SOIL_DEPTH_RANGE_M,
        )
        cache_ok, cache_reason = land_skill.land_skill_cache_status(
            outfile, expected_attrs, cohorts
        )
        if cache_ok and not RUN["force_compute"]:
            skill = load_netcdf(outfile)
            print(f"Loading compatible skill cache: {outfile}")
        else:
            if outfile.exists() and not RUN["force_compute"]:
                print(f"Recomputing incompatible cache {outfile}: {cache_reason}")
            skill = land_skill.compute_land_acc_skill(
                forecast_by_case_month[case_name][init_month],
                valid_time_by_case_month[case_name][init_month],
                reference, climy0, climy1,
                nleads=min(RUN["seasonal_nlead"], len(common_years_by_month[init_month])),
                detrend=RUN["detrend"],
                reference_is_anomaly=reference_is_anomaly,
                target_years_by_lead=common_years_by_month[init_month],
            ).compute()
            skill.attrs.update(expected_attrs)
            skill.attrs.update({
                "reference_product": REFERENCE_PRODUCT,
                "sample_alignment": "complete configured target years shared across E3SM cases and reference by lead",
                "evaluation_protocol": EVALUATION_PROTOCOL,
                "pointwise_significance": "effective-correlation p-value; no spatial multiple-testing correction",
            })
            atomic_to_netcdf(skill, outfile, **netcdf_write_options)
        land_skill.validate_land_skill_dataset(skill, expected_attrs, cohorts)
        skill_by_case_month[case_name][init_month] = skill
        print(outfile, dict(skill.sizes), "N=", skill.sample_count.values.tolist())

## Combined nRMSE figure

This follows the multi-panel style of `1b_atm_leadtime_rmse_skill_map.ipynb`: rows are valid seasonal leads, columns are E3SM cases grouped under the configured initialization-month headers, and only the outer axes carry latitude/longitude labels.


In [ ]:
import cartopy.crs as ccrs

from esp_lab.utils import mapplot_utils as maps
from esp_lab.utils import mov_utils as mov

spec = land_skill.get_land_variable_spec(field)

VARIABLE_DISPLAY_NAME = spec.plot_name
VARIABLE_TITLE_CONTEXT = (
    f"{VARIABLE_DISPLAY_NAME} ({REFERENCE_LEVEL_DEPTH})"
    if field == "H2OSOI" else VARIABLE_DISPLAY_NAME
)
VARIABLE_DIFFERENCE_CONTEXT = (
    f"{VARIABLE_DISPLAY_NAME} ({REFERENCE_LEVEL_DEPTH})"
    if field == "H2OSOI" else VARIABLE_DISPLAY_NAME
)
VARIABLE_CAPTION_NAME = (
    REFERENCE_CAPTION_NAME if field == "H2OSOI"
    else "total water storage anomaly" if field == "TWS"
    else VARIABLE_DISPLAY_NAME.lower()
)

DETREND_DISPLAY_NAME = (
    "linear-detrend" if RUN["detrend"] else "no-detrend"
)
PLOT = FIGURE_SETTINGS["rmse"]

case_display_names = {
    case_name: case_cfg.get("display_name", case_name)
    for case_name, case_cfg in E3SM_CASES.items()
}

column_specs = [
    (init_month, case_name)
    for init_month in RUN["init_months"]
    for case_name in E3SM_CASES
    if init_month in skill_by_case_month.get(case_name, {})
]
FULL_SEASONAL_LEADS = list(range(3, RUN["monthly_nlead"], 3))[
    : min(RUN["seasonal_nlead"], PLOT["max_leads"])
]
plot_leads_by_month = {
    month: FULL_SEASONAL_LEADS for month in RUN["init_months"]
}
if not column_specs or not any(plot_leads_by_month.values()):
    raise RuntimeError("No common case/month/lead combinations are available for plotting")

rmse_by_case_month = {}
for init_month, case_name in column_specs:
    skill = skill_by_case_month[case_name][init_month]
    rmse_by_case_month[(init_month, case_name)] = skill.rmse.where(
        skill.valid_sample_count == skill.sample_count
    )

nrows = max(map(len, plot_leads_by_month.values()))
ncols = len(column_specs)
projection = ccrs.PlateCarree()
fig = plt.figure(figsize=PLOT["figsize"])
mappable = None

for row in range(nrows):
    for col, (init_month, case_name) in enumerate(column_specs):
        month_leads = plot_leads_by_month[init_month]
        subplot = row * ncols + col + 1
        lead = month_leads[row]
        skill = skill_by_case_month[case_name][init_month]
        title = case_display_names.get(case_name, case_name) if row == 0 else ""
        if lead in skill.L.values:
            ax, mappable = maps.map_pcolor_global_subplot(
                fig, rmse_by_case_month[(init_month, case_name)].sel(L=lead),
                skill.lon, skill.lat,
                PLOT["color_interval"], PLOT["color_min"], PLOT["color_max"],
                title, nrows, ncols, subplot, projection,
                cmap=PLOT["cmap"], cutoff=PLOT["color_cutoff"],
                fontsize=PLOT["font_size"],
            )
        else:
            ax = add_missing_map_panel(
                fig, nrows=nrows, ncols=ncols, subplot=subplot, title=title,
                message=f"No {REFERENCE_DISPLAY_NAME}\nobservations",
                font_size=PLOT["font_size"],
            )
        style_global_map_axis(
            ax, row=row, column=col, nrows=nrows,
            label_size=PLOT["font_size"] * PLOT["axis_label_font_scale"],
            style=MAP_STYLE,
        )
        if col == 0 or (col > 0 and column_specs[col - 1][0] != init_month):
            display_lead = int(lead) - 2
            add_lead_badge(
                ax, f"lead {display_lead}: {seasonal_label(init_month, display_lead)}",
                font_size=PLOT["font_size"] * PLOT["lead_label_font_scale"],
            )

skill_figure_title = (
    f"{VARIABLE_TITLE_CONTEXT} nRMSE: E3SM, {DETREND_DISPLAY_NAME} "
    f"(ref: {REFERENCE_DISPLAY_NAME})"
)
fig.suptitle(
    skill_figure_title,
    fontsize=PLOT["font_size"] * PLOT["title_font_scale"],
    fontweight=PLOT["font_weight"], y=PLOT["title_y"],
)
fig.tight_layout(rect=PLOT["layout_rect"])
fig.subplots_adjust(
    bottom=PLOT["subplot_bottom"], hspace=PLOT["subplot_hspace"],
    wspace=PLOT["subplot_wspace"],
)

# Add a small gap and divider between initialization-month groups.
second_month = RUN["init_months"][1] if len(RUN["init_months"]) > 1 else None
if second_month is not None:
    second_cols = [i for i, (month, _) in enumerate(column_specs) if month == second_month]
    for row in range(nrows):
        for col in second_cols:
            ax = fig.axes[row * ncols + col]
            pos = ax.get_position()
            ax.set_position([pos.x0 + PLOT["section_gap"], pos.y0, pos.width, pos.height])

for init_month in RUN["init_months"]:
    cols = [i for i, (month, _) in enumerate(column_specs) if month == init_month]
    if cols:
        left = min(fig.axes[col].get_position().x0 for col in cols)
        right = max(fig.axes[col].get_position().x1 for col in cols)
        fig.text(
            (left + right) / 2, PLOT["month_header_y"],
            f"{init_month_names.get(init_month, init_month)} initialization",
            ha="center", va="bottom", fontsize=PLOT["font_size"],
            fontweight=PLOT["font_weight"],
        )

if second_month is not None:
    first_cols = [i for i, (month, _) in enumerate(column_specs) if month != second_month]
    second_cols = [i for i, (month, _) in enumerate(column_specs) if month == second_month]
    divider_x = (
        max(fig.axes[col].get_position().x1 for col in first_cols)
        + min(fig.axes[col].get_position().x0 for col in second_cols)
    ) / 2
    fig.add_artist(plt.Line2D(
        [divider_x, divider_x],
        [PLOT["divider_y0"], PLOT["divider_y1"]],
        transform=fig.transFigure,
        color=PLOT["divider_color"], linewidth=PLOT["divider_linewidth"],
    ))

colorbar_ax = fig.add_axes(PLOT["colorbar_rect"])
colorbar = fig.colorbar(
    mappable, cax=colorbar_ax, orientation=PLOT["colorbar_orientation"]
)
colorbar.set_label(
    PLOT["colorbar_label"], fontsize=PLOT["font_size"],
    fontweight=PLOT["font_weight"],
)
colorbar.ax.tick_params(
    labelsize=PLOT["font_size"] * PLOT["colorbar_tick_font_scale"]
)

figure_path = FIGURE_OUTDIR / PLOT["filename"]
mov.save_figure(
    fig, figure_path, mode="", metric="leadtime_rmse",
    title=skill_figure_title,
    caption=(f"Normalized RMSE rows by seasonal lead using complete configured cohorts for "
             f"initializations {PROCESSING_YEARS[0]}\u2013{PROCESSING_YEARS[1]} "
             f"and climatology {climy0}\u2013{climy1}; "
             f"{'linear detrending' if RUN['detrend'] else 'no detrending'}."),
    dpi=PLOT["dpi"],
)
print("Saved figure:", figure_path)
plt.show()


## nRMSE differences relative to a selected control

Set `RMSE_DIFFERENCE_CONTROL` in **User setup** to any configured experiment. Each panel shows another experiment minus that control, so positive values indicate higher (worse) nRMSE than the selected control. The control itself is omitted.


In [ ]:
DIFF_PLOT = FIGURE_SETTINGS["difference"]
if RMSE_DIFFERENCE_CONTROL not in E3SM_CASES:
    raise ValueError(
        f"RMSE_DIFFERENCE_CONTROL={RMSE_DIFFERENCE_CONTROL!r} is not in E3SM_CASES; "
        f"choose one of {list(E3SM_CASES)}"
    )
if RMSE_DIFFERENCE_CONTROL not in skill_by_case_month:
    raise RuntimeError(
        f"No cached RMSE was loaded for control {RMSE_DIFFERENCE_CONTROL!r}"
    )

difference_cases = [
    case_name for case_name in E3SM_CASES
    if case_name != RMSE_DIFFERENCE_CONTROL
]
if not difference_cases:
    raise RuntimeError("At least two E3SM cases are required for nRMSE differences")

difference_column_specs = [
    (init_month, case_name)
    for init_month in RUN["init_months"]
    for case_name in difference_cases
    if init_month in skill_by_case_month.get(RMSE_DIFFERENCE_CONTROL, {})
    and init_month in skill_by_case_month.get(case_name, {})
]
difference_months = list(dict.fromkeys(
    init_month for init_month, _ in difference_column_specs
))
if not difference_column_specs or not FULL_SEASONAL_LEADS:
    raise RuntimeError(
        "No common experiment/control month/lead combinations are available "
        "for the nRMSE differences"
    )

rmse_difference = {}
for init_month, case_name in difference_column_specs:
    experiment_skill = skill_by_case_month[case_name][init_month]
    control_skill = skill_by_case_month[RMSE_DIFFERENCE_CONTROL][init_month]
    experiment_rmse = experiment_skill.rmse.where(
        experiment_skill.valid_sample_count == experiment_skill.sample_count
    )
    control_rmse = control_skill.rmse.where(
        control_skill.valid_sample_count == control_skill.sample_count
    )
    rmse_difference[(init_month, case_name)] = experiment_rmse - control_rmse

nrows = len(FULL_SEASONAL_LEADS)
ncols = len(difference_column_specs)
section_gap = DIFF_PLOT["section_gap"] if len(difference_months) > 1 else 0.0
fig = plt.figure(figsize=DIFF_PLOT["figsize"])
mappable = None
for row in range(nrows):
    for col, (init_month, case_name) in enumerate(difference_column_specs):
        subplot = row * ncols + col + 1
        lead = FULL_SEASONAL_LEADS[row]
        skill = skill_by_case_month[case_name][init_month]
        title = case_display_names.get(case_name, case_name) if row == 0 else ""
        difference = rmse_difference[(init_month, case_name)]
        if lead in difference.L.values:
            ax, mappable = maps.map_pcolor_global_subplot(
                fig, difference.sel(L=lead), skill.lon, skill.lat,
                DIFF_PLOT["color_interval"], DIFF_PLOT["color_min"], DIFF_PLOT["color_max"],
                title, nrows, ncols, subplot, projection,
                cmap=DIFF_PLOT["cmap"], cutoff=DIFF_PLOT["color_cutoff"],
                fontsize=DIFF_PLOT["font_size"],
            )
        else:
            ax = add_missing_map_panel(
                fig, nrows=nrows, ncols=ncols, subplot=subplot, title=title,
                message=f"No {REFERENCE_DISPLAY_NAME}\nobservations",
                font_size=DIFF_PLOT["font_size"],
            )
        style_global_map_axis(
            ax, row=row, column=col, nrows=nrows,
            label_size=(
                DIFF_PLOT["font_size"] * DIFF_PLOT["axis_label_font_scale"]
            ),
            style=MAP_STYLE,
        )
        if col == 0 or difference_column_specs[col - 1][0] != init_month:
            display_lead = int(lead) - 2
            add_lead_badge(
                ax, f"lead {display_lead}: {seasonal_label(init_month, display_lead)}",
                font_size=(
                    DIFF_PLOT["font_size"] * DIFF_PLOT["lead_label_font_scale"]
                ),
            )

difference_title = DIFF_PLOT["title_template"].format(
    variable=VARIABLE_DIFFERENCE_CONTEXT,
    reference=REFERENCE_DISPLAY_NAME,
    control=case_display_names[RMSE_DIFFERENCE_CONTROL],
    trend=DETREND_DISPLAY_NAME,
)
difference_caption = (
    f"nRMSE differences (experiment minus "
    f"{case_display_names[RMSE_DIFFERENCE_CONTROL]}) for {VARIABLE_CAPTION_NAME} relative "
    f"to {REFERENCE_DISPLAY_NAME}, by seasonal lead and initialization month; "
    f"initializations {PROCESSING_YEARS[0]}–{PROCESSING_YEARS[1]}, "
    f"climatology {climy0}–{climy1}, {DETREND_DISPLAY_NAME}."
)
fig.suptitle(
    difference_title,
    fontsize=DIFF_PLOT["font_size"] * DIFF_PLOT["title_font_scale"],
    fontweight=DIFF_PLOT["font_weight"], y=DIFF_PLOT["title_y"],
)
layout_left, layout_bottom, layout_right, layout_top = DIFF_PLOT["layout_edges"]
layout_right -= section_gap * (len(difference_months) - 1)
fig.tight_layout(
    rect=(layout_left, layout_bottom, layout_right, layout_top),
    pad=DIFF_PLOT["layout_pad"],
    h_pad=DIFF_PLOT["layout_h_pad"],
    w_pad=DIFF_PLOT["layout_w_pad"],
)
fig.subplots_adjust(
    top=DIFF_PLOT["subplot_top"],
    bottom=DIFF_PLOT["subplot_bottom"],
    hspace=DIFF_PLOT["subplot_hspace"],
    wspace=DIFF_PLOT["subplot_wspace"],
)

# Separate and label initialization-month column groups.
for group_index, init_month in enumerate(difference_months):
    if group_index == 0:
        continue
    month_cols = [
        col for col, (month, _) in enumerate(difference_column_specs)
        if month == init_month
    ]
    for row in range(nrows):
        for col in month_cols:
            ax = fig.axes[row * ncols + col]
            pos = ax.get_position()
            ax.set_position([
                pos.x0 + group_index * section_gap,
                pos.y0, pos.width, pos.height,
            ])

month_header_y = DIFF_PLOT["month_header_y"]
for init_month in difference_months:
    cols = [
        col for col, (month, _) in enumerate(difference_column_specs)
        if month == init_month
    ]
    left = min(fig.axes[col].get_position().x0 for col in cols)
    right = max(fig.axes[col].get_position().x1 for col in cols)
    fig.text(
        (left + right) / 2, month_header_y,
        f"{init_month_names.get(init_month, init_month)} initialization",
        ha="center", va="bottom",
        fontsize=DIFF_PLOT["font_size"] * DIFF_PLOT["month_header_font_scale"],
        fontweight=DIFF_PLOT["font_weight"],
    )

for left_month, right_month in zip(difference_months, difference_months[1:]):
    left_cols = [
        col for col, (month, _) in enumerate(difference_column_specs)
        if month == left_month
    ]
    right_cols = [
        col for col, (month, _) in enumerate(difference_column_specs)
        if month == right_month
    ]
    divider_x = (
        max(fig.axes[col].get_position().x1 for col in left_cols)
        + min(fig.axes[col].get_position().x0 for col in right_cols)
    ) / 2
    fig.add_artist(plt.Line2D(
        [divider_x, divider_x],
        [DIFF_PLOT["divider_y0"], DIFF_PLOT["divider_y1"]],
        transform=fig.transFigure, color=DIFF_PLOT["divider_color"],
        linewidth=DIFF_PLOT["divider_linewidth"],
    ))
colorbar_ax = fig.add_axes(DIFF_PLOT["colorbar_rect"])
colorbar = fig.colorbar(
    mappable, cax=colorbar_ax, orientation=DIFF_PLOT["colorbar_orientation"]
)
colorbar.set_label(
    DIFF_PLOT["colorbar_label"],
    fontsize=DIFF_PLOT["font_size"],
    fontweight=DIFF_PLOT["font_weight"],
)
colorbar.ax.tick_params(
    labelsize=(
        DIFF_PLOT["font_size"] * DIFF_PLOT["colorbar_tick_font_scale"]
    )
)

difference_path = FIGURE_OUTDIR / DIFF_PLOT["filename"]
mov.save_figure(
    fig, difference_path, mode="", metric="leadtime_rmse_diff",
    title=difference_title,
    caption=difference_caption,
    dpi=DIFF_PLOT["dpi"],
)
print("Saved difference figure:", difference_path)
plt.show()


## Cleanup

Prepared inputs, skills, and figures remain on disk. Run this after completion or an
interrupted calculation to close staged datasets, the Dask client, and its cluster.


In [ ]:
close_notebook_resources(globals())
print("Closed land-workflow datasets and Dask resources.")
